# Chapter 5. Evaluation
## 5.2.3 Oracle-Parameter Search

This notebook selects the highest-PSNR primary parameter for each test image, noise level, and denoiser. SSIM is then measured from the selected output.

In [3]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_dir) not in sys.path:
    sys.path.append(str(project_dir))

from src.denoising_config import (
    AD_FIXED_SETTINGS, AD_K_VALUES, BF_FIXED_SETTINGS, BF_SIGMA_COLOR_VALUES,
    METHOD_LABELS, METHOD_ORDER, NLM_FIXED_SETTINGS, NLM_H_VALUES,
    PRIMARY_PARAMETER_NAMES, TV_WEIGHT_VALUES, denoise_with_parameters
)
from src.noise import add_gaussian_noise
from src.utils import load_grayscale_image

In [5]:
noise_levels = list(range(2, 16))
seed = 42
num_test_images = 100
n_jobs = 24
force_rerun = False

test_dir = project_dir / "data" / "test"
result_dir = project_dir / "results" / "Chapter5_SearchOracleParameter"
result_dir.mkdir(parents=True, exist_ok=True)

calibrated_values = {
    "AD": AD_K_VALUES, "TV": TV_WEIGHT_VALUES,
    "BF": BF_SIGMA_COLOR_VALUES, "NLM": NLM_H_VALUES
}
search_steps = {"AD": 0.01, "TV": 0.0025, "BF": 0.01, "NLM": 0.0025}

candidate_values = {}
for method in METHOD_ORDER:
    lower = float(np.min(calibrated_values[method]))
    upper = float(np.max(calibrated_values[method]))
    step = search_steps[method]
    candidate_values[method] = np.round(np.arange(lower, upper + step / 2, step), 4)

print("Test directory:", test_dir)
print("Noise levels (%):", noise_levels)
for method in METHOD_ORDER:
    values = candidate_values[method]
    print(f"{method}: {len(values)} candidates from {values[0]:.4f} to {values[-1]:.4f}")

Test directory: D:\MScProject\data\test
Noise levels (%): [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
AD: 22 candidates from 0.0200 to 0.2300
TV: 50 candidates from 0.0100 to 0.1325
BF: 21 candidates from 0.0200 to 0.2200
NLM: 56 candidates from 0.0225 to 0.1600


In [4]:
image_files = []
for pattern in ["*.png", "*.jpg", "*.jpeg", "*.bmp", "*.tif", "*.tiff"]:
    image_files.extend(test_dir.rglob(pattern))

image_files = sorted(set(image_files))
if len(image_files) != num_test_images:
    raise FileNotFoundError(
        f"Expected {num_test_images} test images, but found {len(image_files)} in {test_dir}"
    )

test_images = [(path.stem, load_grayscale_image(path)) for path in image_files]

for image_name, clean_image in test_images:
    if clean_image.ndim != 2:
        raise ValueError(f"{image_name} is not a grayscale image")
    if clean_image.min() < 0 or clean_image.max() > 1:
        raise ValueError(f"{image_name} is not normalised to [0, 1]")

print("Number of test images:", len(test_images))

Number of test images: 100


## Oracle search

The same candidate grid is tested for each case. The candidate with the highest PSNR is selected.

In [5]:
def make_case_seed(image_index, noise_percent):
    noise_index = noise_levels.index(noise_percent)
    return seed + noise_index * num_test_images + image_index


def make_parameters(method, value):
    value = float(value)

    if method == "AD":
        parameters = AD_FIXED_SETTINGS.copy()
        parameters["kappa"] = value
        return parameters
    if method == "TV":
        return {"weight": value}
    if method == "BF":
        parameters = BF_FIXED_SETTINGS.copy()
        parameters["sigma_color"] = value
        return parameters

    parameters = NLM_FIXED_SETTINGS.copy()
    parameters["h"] = value
    return parameters


def evaluate_candidate_case(method, image_index, image_name, clean_image, noise_percent):
    noisy_image = add_gaussian_noise(
        clean_image, sigma=noise_percent / 100,
        seed=make_case_seed(image_index, noise_percent)
    ).astype(np.float32)

    rows = []
    for value in candidate_values[method]:
        parameters = make_parameters(method, value)
        denoised_image = denoise_with_parameters(noisy_image, method, parameters)

        if denoised_image.shape != clean_image.shape:
            raise ValueError(f"{method} returned an unexpected shape")
        if not np.isfinite(denoised_image).all():
            raise ValueError(f"{method} returned a non-finite value")

        psnr = peak_signal_noise_ratio(clean_image, denoised_image, data_range=1.0)
        rows.append({
            "method": method, "image": image_name, "image_index": image_index,
            "true_noise_percent": noise_percent, "parameter_value": float(value),
            "psnr": psnr
        })

    return rows

In [6]:
def save_csv_safely(data, output_path):
    temporary_path = output_path.with_name(output_path.stem + "_temporary.csv")
    data.to_csv(temporary_path, index=False)
    temporary_path.replace(output_path)


def reached_boundary(method, value):
    values = candidate_values[method]
    return np.isclose(value, values[0]) or np.isclose(value, values[-1])


def evaluate_selected_case(row):
    clean_image = test_images[int(row.image_index)][1]
    noisy_image = add_gaussian_noise(
        clean_image, sigma=row.true_noise_percent / 100,
        seed=make_case_seed(int(row.image_index), row.true_noise_percent)
    ).astype(np.float32)

    parameters = make_parameters(row.method, row.parameter_value)
    denoised_image = denoise_with_parameters(noisy_image, row.method, parameters)

    if denoised_image.shape != clean_image.shape:
        raise ValueError(f"{row.method} returned an unexpected shape")
    if not np.isfinite(denoised_image).all():
        raise ValueError(f"{row.method} returned a non-finite value")

    psnr_value = peak_signal_noise_ratio(clean_image, denoised_image, data_range=1.0)
    ssim_value = structural_similarity(clean_image, denoised_image, data_range=1.0)

    if not np.isclose(psnr_value, row.psnr, rtol=0, atol=1e-9):
        raise RuntimeError("The repeated PSNR does not match the search result")

    return {
        "image": row.image, "image_index": int(row.image_index),
        "true_noise_percent": row.true_noise_percent, "method": row.method,
        "strategy": "Oracle", "parameter_name": PRIMARY_PARAMETER_NAMES[row.method],
        "parameter_value": row.parameter_value,
        "at_search_boundary": row.at_search_boundary,
        "psnr": psnr_value, "ssim": ssim_value
    }


def run_oracle_search(method):
    if method not in METHOD_ORDER:
        raise ValueError("Unknown denoising method")

    candidate_path = result_dir / f"oracle_candidate_search_{method}.csv"
    oracle_path = result_dir / f"oracle_results_{method}.csv"
    expected_cases = num_test_images * len(noise_levels)

    if oracle_path.exists() and not force_rerun:
        oracle_results = pd.read_csv(oracle_path)
        if len(oracle_results) != expected_cases:
            raise RuntimeError(f"The saved {method} result is incomplete")
        if set(oracle_results["method"]) != {method}:
            raise ValueError(f"The saved result does not contain only {method}")
        if not np.isfinite(oracle_results[["psnr", "ssim"]]).all().all():
            raise ValueError(f"The saved {method} result is not finite")
        print(f"{METHOD_LABELS[method]} is already complete.")
        return oracle_results

    search_was_run = False
    if candidate_path.exists() and not force_rerun:
        search_results = pd.read_csv(candidate_path)
        print(f"Loaded the saved {method} candidate search.")
    else:
        search_outputs = Parallel(n_jobs=n_jobs, prefer="threads", verbose=5)(
            delayed(evaluate_candidate_case)(
                method, image_index, image_name, clean_image, noise_percent
            )
            for noise_percent in noise_levels
            for image_index, (image_name, clean_image) in enumerate(test_images)
        )
        search_results = pd.DataFrame([row for case_rows in search_outputs for row in case_rows])
        search_was_run = True

    expected_rows = expected_cases * len(candidate_values[method])
    if len(search_results) != expected_rows:
        raise RuntimeError(
            f"Expected {expected_rows} {method} results, but obtained {len(search_results)}"
        )
    if set(search_results["method"]) != {method}:
        raise ValueError(f"The saved search does not contain only {method}")
    if not np.isfinite(search_results[["parameter_value", "psnr"]]).all().all():
        raise ValueError(f"The {method} search contains a non-finite value")

    if search_was_run:
        save_csv_safely(search_results, candidate_path)
        print(f"Saved the {method} candidate search.")

    case_columns = ["image_index", "true_noise_percent"]
    best_indices = search_results.groupby(case_columns)["psnr"].idxmax()
    selected_search = search_results.loc[best_indices].copy()
    selected_search = selected_search.sort_values(
        ["true_noise_percent", "image_index"]
    ).reset_index(drop=True)
    selected_search["at_search_boundary"] = [
        reached_boundary(method, value) for value in selected_search["parameter_value"]
    ]

    if len(selected_search) != expected_cases:
        raise RuntimeError(f"Expected {expected_cases} selected {method} cases")

    selected_rows = list(selected_search.itertuples(index=False))
    oracle_outputs = Parallel(n_jobs=n_jobs, prefer="threads", verbose=5)(
        delayed(evaluate_selected_case)(row) for row in selected_rows
    )

    oracle_results = pd.DataFrame(oracle_outputs)
    oracle_results = oracle_results.sort_values(
        ["true_noise_percent", "image_index"]
    ).reset_index(drop=True)

    if len(oracle_results) != expected_cases:
        raise RuntimeError(f"Expected {expected_cases} Oracle results for {method}")
    if not np.isfinite(oracle_results[["psnr", "ssim"]]).all().all():
        raise ValueError(f"The {method} Oracle result is not finite")

    save_csv_safely(oracle_results, oracle_path)
    print(f"Completed and saved {METHOD_LABELS[method]}.")
    print("Selected cases:", len(oracle_results))
    print("Boundary cases:", int(oracle_results["at_search_boundary"].sum()))
    return oracle_results

## Run the four denoisers

In [6]:
# 1. Anisotropic diffusion
ad_oracle_results = run_oracle_search("AD")

[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done  24 tasks      | elapsed:    6.7s
[Parallel(n_jobs=24)]: Done 114 tasks      | elapsed:   32.0s
[Parallel(n_jobs=24)]: Done 240 tasks      | elapsed:  1.1min
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:  1.8min
[Parallel(n_jobs=24)]: Done 600 tasks      | elapsed:  2.6min
[Parallel(n_jobs=24)]: Done 834 tasks      | elapsed:  3.6min
[Parallel(n_jobs=24)]: Done 1104 tasks      | elapsed:  4.7min
[Parallel(n_jobs=24)]: Done 1400 out of 1400 | elapsed:  6.0min finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.


Saved the AD candidate search.


[Parallel(n_jobs=24)]: Done  24 tasks      | elapsed:    0.4s
[Parallel(n_jobs=24)]: Done 114 tasks      | elapsed:    2.1s
[Parallel(n_jobs=24)]: Done 240 tasks      | elapsed:    4.5s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:    7.3s
[Parallel(n_jobs=24)]: Done 600 tasks      | elapsed:   10.8s
[Parallel(n_jobs=24)]: Done 834 tasks      | elapsed:   15.1s
[Parallel(n_jobs=24)]: Done 1104 tasks      | elapsed:   19.9s


Completed and saved Anisotropic diffusion.
Selected cases: 1400
Boundary cases: 228


[Parallel(n_jobs=24)]: Done 1400 out of 1400 | elapsed:   24.8s finished


In [7]:
# 2. Total variation
tv_oracle_results = run_oracle_search("TV")

[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done  24 tasks      | elapsed:   21.6s
[Parallel(n_jobs=24)]: Done 114 tasks      | elapsed:  1.5min
[Parallel(n_jobs=24)]: Done 240 tasks      | elapsed:  3.0min
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:  5.0min
[Parallel(n_jobs=24)]: Done 600 tasks      | elapsed:  7.5min
[Parallel(n_jobs=24)]: Done 834 tasks      | elapsed: 10.9min
[Parallel(n_jobs=24)]: Done 1104 tasks      | elapsed: 15.3min
[Parallel(n_jobs=24)]: Done 1400 out of 1400 | elapsed: 20.0min finished


Saved the TV candidate search.


[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done  24 tasks      | elapsed:    0.5s
[Parallel(n_jobs=24)]: Done 114 tasks      | elapsed:    2.2s
[Parallel(n_jobs=24)]: Done 240 tasks      | elapsed:    6.6s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:   10.6s
[Parallel(n_jobs=24)]: Done 600 tasks      | elapsed:   15.8s
[Parallel(n_jobs=24)]: Done 834 tasks      | elapsed:   22.4s
[Parallel(n_jobs=24)]: Done 1104 tasks      | elapsed:   30.2s


Completed and saved Total variation.
Selected cases: 1400
Boundary cases: 158


[Parallel(n_jobs=24)]: Done 1400 out of 1400 | elapsed:   39.3s finished


In [8]:
# 3. Bilateral filtering
bf_oracle_results = run_oracle_search("BF")

[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done  24 tasks      | elapsed:   51.1s
[Parallel(n_jobs=24)]: Done 114 tasks      | elapsed:  2.8min
[Parallel(n_jobs=24)]: Done 240 tasks      | elapsed:  5.9min
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:  9.6min
[Parallel(n_jobs=24)]: Done 600 tasks      | elapsed: 14.4min
[Parallel(n_jobs=24)]: Done 834 tasks      | elapsed: 19.6min
[Parallel(n_jobs=24)]: Done 1104 tasks      | elapsed: 25.8min
[Parallel(n_jobs=24)]: Done 1400 out of 1400 | elapsed: 32.4min finished
[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.


Saved the BF candidate search.


[Parallel(n_jobs=24)]: Done  24 tasks      | elapsed:    2.1s
[Parallel(n_jobs=24)]: Done 114 tasks      | elapsed:    9.6s
[Parallel(n_jobs=24)]: Done 240 tasks      | elapsed:   19.1s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:   30.8s
[Parallel(n_jobs=24)]: Done 600 tasks      | elapsed:   43.0s
[Parallel(n_jobs=24)]: Done 834 tasks      | elapsed:   60.0s
[Parallel(n_jobs=24)]: Done 1104 tasks      | elapsed:  1.3min


Completed and saved Bilateral filtering.
Selected cases: 1400
Boundary cases: 171


[Parallel(n_jobs=24)]: Done 1400 out of 1400 | elapsed:  1.7min finished


In [9]:
# 4. Non-local means
nlm_oracle_results = run_oracle_search("NLM")

[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done  24 tasks      | elapsed:    6.7s
[Parallel(n_jobs=24)]: Done 114 tasks      | elapsed:   30.0s
[Parallel(n_jobs=24)]: Done 240 tasks      | elapsed:   59.0s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:  1.6min
[Parallel(n_jobs=24)]: Done 600 tasks      | elapsed:  2.3min
[Parallel(n_jobs=24)]: Done 834 tasks      | elapsed:  3.2min
[Parallel(n_jobs=24)]: Done 1104 tasks      | elapsed:  4.1min
[Parallel(n_jobs=24)]: Done 1400 out of 1400 | elapsed:  5.1min finished


Saved the NLM candidate search.


[Parallel(n_jobs=24)]: Using backend ThreadingBackend with 24 concurrent workers.
[Parallel(n_jobs=24)]: Done  24 tasks      | elapsed:    0.1s
[Parallel(n_jobs=24)]: Done 114 tasks      | elapsed:    0.9s
[Parallel(n_jobs=24)]: Done 240 tasks      | elapsed:    1.9s
[Parallel(n_jobs=24)]: Done 402 tasks      | elapsed:    3.2s
[Parallel(n_jobs=24)]: Done 600 tasks      | elapsed:    4.9s
[Parallel(n_jobs=24)]: Done 834 tasks      | elapsed:    6.8s
[Parallel(n_jobs=24)]: Done 1104 tasks      | elapsed:    9.0s


Completed and saved Non-local means.
Selected cases: 1400
Boundary cases: 165


[Parallel(n_jobs=24)]: Done 1400 out of 1400 | elapsed:   11.3s finished


## Combined results

In [10]:
oracle_paths = {
    method: result_dir / f"oracle_results_{method}.csv" for method in METHOD_ORDER
}
missing_methods = [method for method, path in oracle_paths.items() if not path.exists()]
if missing_methods:
    raise FileNotFoundError("Run these Oracle searches first: " + ", ".join(missing_methods))

oracle_results = pd.concat(
    [pd.read_csv(oracle_paths[method]) for method in METHOD_ORDER], ignore_index=True
)
oracle_results["at_search_boundary"] = (
    oracle_results["at_search_boundary"].astype(str).map({"True": True, "False": False})
)
if oracle_results["at_search_boundary"].isna().any():
    raise ValueError("A boundary flag could not be read")

oracle_results = oracle_results.sort_values(
    ["method", "true_noise_percent", "image_index"]
).reset_index(drop=True)

expected_total = num_test_images * len(noise_levels) * len(METHOD_ORDER)
if len(oracle_results) != expected_total:
    raise RuntimeError(f"Expected {expected_total} combined results, but obtained {len(oracle_results)}")

print("Combined Oracle results:", len(oracle_results))
display(oracle_results.head())

Combined Oracle results: 5600


,image,image_index,true_noise_percent,method,strategy,parameter_name,parameter_value,at_search_boundary,psnr,ssim
0,1,0,2,AD,Oracle,kappa,0.02,True,34.034735,0.952916
1,2,1,2,AD,Oracle,kappa,0.02,True,37.291542,0.955519
2,3,2,2,AD,Oracle,kappa,0.02,True,37.171387,0.945708
3,4,3,2,AD,Oracle,kappa,0.02,True,37.324663,0.957255
4,5,4,2,AD,Oracle,kappa,0.02,True,35.396330,0.960659


In [11]:
oracle_level_summary = (
    oracle_results.groupby(["method", "true_noise_percent"], as_index=False)
    .agg(
        psnr=("psnr", "mean"), ssim=("ssim", "mean"),
        mean_parameter=("parameter_value", "mean"),
        median_parameter=("parameter_value", "median")
    )
)

oracle_method_summary = (
    oracle_results.groupby("method", as_index=False)
    .agg(
        overall_mean_psnr=("psnr", "mean"),
        overall_mean_ssim=("ssim", "mean"),
        boundary_cases=("at_search_boundary", "sum")
    )
    .set_index("method").loc[METHOD_ORDER].reset_index()
)

oracle_method_summary["boundary_percent"] = (
    100 * oracle_method_summary["boundary_cases"] / (num_test_images * len(noise_levels))
)
oracle_method_summary["Method"] = oracle_method_summary["method"].map(METHOD_LABELS)

summary_table = oracle_method_summary[[
    "Method", "overall_mean_psnr", "overall_mean_ssim",
    "boundary_cases", "boundary_percent"
]].rename(columns={
    "overall_mean_psnr": "Oracle PSNR (dB)",
    "overall_mean_ssim": "Oracle SSIM",
    "boundary_cases": "Boundary cases",
    "boundary_percent": "Boundary cases (%)"
})

display(summary_table.round({
    "Oracle PSNR (dB)": 3, "Oracle SSIM": 4, "Boundary cases (%)": 2
}))

,Method,Oracle PSNR (dB),Oracle SSIM,Boundary cases,Boundary cases (%)
0,Anisotropic diffusion,28.803,0.7729,228,16.29
1,Total variation,29.056,0.7959,158,11.29
2,Bilateral filtering,26.809,0.6784,171,12.21
3,Non-local means,29.042,0.7782,165,11.79


In [ ]:
save_csv_safely(oracle_results, result_dir / "oracle_results.csv")
save_csv_safely(oracle_level_summary, result_dir / "oracle_level_summary.csv")
save_csv_safely(oracle_method_summary, result_dir / "oracle_method_summary.csv")

print("Combined results saved to:", result_dir)